# Intelligent Pharma-Context Engine - Demo Notebook

This notebook demonstrates the full end-to-end pipeline:
1. **Stage 1**: Image preprocessing, OCR, and barcode detection
2. **Stage 2**: Entity extraction and verification against OpenFDA/RxNorm
3. **Stage 3**: Clinical enrichment via Gemini LLM

## Output
The pipeline produces an enriched JSON record containing:
- Canonical drug name
- Manufacturer
- Composition (active ingredients)
- Dosage/strength
- Clinical metadata (storage, warnings, side effects)

In [ ]:
# Setup: Load environment variables
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

print(f"Project root: {project_root}")
print(f"GEMINI_API_KEY set: {'Yes' if os.getenv('GEMINI_API_KEY') else 'No'}")

## 1. Load and Display Sample Image

In [ ]:
import matplotlib.pyplot as plt
import cv2
from glob import glob

# Find a sample image from the dataset
data_dir = project_root / 'data' / 'raw' / 'medicine bottle.v1i.yolov12'
sample_images = list(data_dir.glob('test/images/*.jpg'))[:5]

if sample_images:
    print(f"Found {len(sample_images)} test images")
    sample_image = str(sample_images[0])
    print(f"Using: {sample_image}")
    
    # Display the image
    img = cv2.imread(sample_image)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title('Sample Medicine Bottle Image')
    plt.axis('off')
    plt.show()
else:
    print("No sample images found. Please add images to data/raw/")
    sample_image = None

## 2. Stage 1: Preprocessing

Apply image enhancement techniques:
- Shadow removal
- Bilateral filtering (noise reduction)
- CLAHE (Contrast Limited Adaptive Histogram Equalization)

In [ ]:
from src.stages.preprocessing import preprocess_pipeline

if sample_image:
    # Run preprocessing
    stages = preprocess_pipeline(sample_image)
    
    # Visualize preprocessing stages
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    stages_to_show = [
        ('original', 'Original'),
        ('normalized', 'Shadow Removed'),
        ('denoised', 'Bilateral Filtered'),
        ('gray', 'Grayscale'),
        ('enhanced_gray', 'CLAHE Enhanced'),
    ]
    
    for idx, (key, title) in enumerate(stages_to_show):
        ax = axes[idx // 3, idx % 3]
        img = stages[key]
        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    
    axes[1, 2].axis('off')  # Hide empty subplot
    plt.tight_layout()
    plt.show()
    
    enhanced_image = stages['enhanced_gray']
    print(f"Enhanced image shape: {enhanced_image.shape}")

## 3. Stage 1: OCR Extraction

Extract text using Tesseract OCR with bounding boxes.

In [ ]:
from src.stages.ocr import run_ocr, extract_text_blocks

if sample_image:
    # Run OCR
    raw_text = run_ocr(enhanced_image)
    text_blocks = extract_text_blocks(enhanced_image, min_conf=30)
    
    print("=" * 50)
    print("RAW OCR TEXT:")
    print("=" * 50)
    print(raw_text)
    print("\n")
    
    print("=" * 50)
    print(f"DETECTED TEXT BLOCKS ({len(text_blocks)} blocks):")
    print("=" * 50)
    for block in text_blocks[:10]:  # Show first 10
        print(f"  [{block['conf']:3d}%] {block['text']!r}")

### Visualize OCR Bounding Boxes

In [ ]:
if sample_image and text_blocks:
    # Draw bounding boxes on image
    img_with_boxes = cv2.cvtColor(stages['original'].copy(), cv2.COLOR_BGR2RGB)
    
    for block in text_blocks:
        x1, y1, x2, y2 = block['bbox']
        conf = block['conf']
        color = (0, 255, 0) if conf > 70 else (255, 165, 0) if conf > 50 else (255, 0, 0)
        cv2.rectangle(img_with_boxes, (x1, y1), (x2, y2), color, 2)
    
    plt.figure(figsize=(12, 10))
    plt.imshow(img_with_boxes)
    plt.title('OCR Text Detections (Green=High Conf, Orange=Medium, Red=Low)')
    plt.axis('off')
    plt.show()

## 4. Stage 1: Barcode Detection

Detect and decode any barcodes/DataMatrix codes on the packaging.

In [ ]:
from src.stages.barcode import decode_all, parse_gs1_barcode

if sample_image:
    barcodes = decode_all(stages['original'])
    
    print(f"Detected {len(barcodes)} barcode(s)")
    for bc in barcodes:
        print(f"  Type: {bc['type']}")
        print(f"  Data: {bc['data']}")
        parsed = parse_gs1_barcode(bc['data'])
        if parsed:
            print(f"  Parsed: {parsed}")
        print()

## 5. Run Full Pipeline

Execute the complete end-to-end pipeline including:
- Stage 1: Detection & Extraction
- Stage 2: Verification (OpenFDA/RxNorm)
- Stage 3: Enrichment (Gemini LLM)

In [ ]:
from src import PharmaContextPipeline

# Initialize pipeline
pipeline = PharmaContextPipeline()
print("Pipeline initialized successfully!")

In [ ]:
if sample_image:
    # Process the image
    result = pipeline.process_image(sample_image)
    
    if result:
        print("Pipeline completed successfully!")
        print(f"Processing time: {result.metrics.processing_time_ms:.0f}ms")
    else:
        print("Pipeline failed to process image.")

## 6. View Enriched JSON Output

The final output contains all extracted, verified, and enriched data.

In [ ]:
import json

if result:
    # Pretty print the full JSON output
    output_json = result.model_dump_json(indent=2)
    print(output_json)

### Key Fields Extracted

In [ ]:
if result:
    print("=" * 60)
    print("EXTRACTED ENTITIES")
    print("=" * 60)
    entities = result.extracted_entities
    print(f"  Drug Name:    {entities.drug_name}")
    print(f"  Generic Name: {entities.generic_name}")
    print(f"  Manufacturer: {entities.manufacturer}")
    print(f"  Strength:     {entities.strength}")
    print(f"  Dosage Form:  {entities.dosage_form}")
    print(f"  Composition:  {entities.composition}")
    
    print("\n" + "=" * 60)
    print("VERIFICATION STATUS")
    print("=" * 60)
    verification = result.verification
    print(f"  Confidence Score: {verification.confidence_score:.2f}")
    print(f"  OpenFDA Found:    {verification.openfda.is_found}")
    print(f"  RxNorm Found:     {verification.rxnorm.is_found}")
    print(f"  Human Review:     {verification.human_review_needed}")
    if verification.openfda.is_found:
        print(f"  FDA Brand Name:   {verification.openfda.brand_name}")
        print(f"  FDA NDC:          {verification.openfda.product_ndc}")
    if verification.rxnorm.is_found:
        print(f"  RxCUI:            {verification.rxnorm.rxcui}")
    
    print("\n" + "=" * 60)
    print("CLINICAL ENRICHMENT")
    print("=" * 60)
    clinical = result.clinical_enrichment
    print(f"  Indications:      {clinical.indications[:2]}..." if len(clinical.indications) > 2 else f"  Indications:      {clinical.indications}")
    print(f"  Warnings:         {clinical.warnings[:2]}..." if len(clinical.warnings) > 2 else f"  Warnings:         {clinical.warnings}")
    print(f"  Side Effects:     {clinical.side_effects[:3]}..." if len(clinical.side_effects) > 3 else f"  Side Effects:     {clinical.side_effects}")
    print(f"  Storage:          {clinical.storage}")
    if clinical.human_summary:
        print(f"\n  Summary: {clinical.human_summary}")

## 7. Save Output to File

In [ ]:
if result:
    output_path = project_root / 'data' / 'processed' / f'{result.image_id}_enriched.json'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'w') as f:
        f.write(result.model_dump_json(indent=2))
    
    print(f"Output saved to: {output_path}")

## 8. Evaluation Metrics

Demonstrate CER (Character Error Rate) and Entity Match Rate calculations.

In [ ]:
from src.evaluation.metrics import calculate_cer, calculate_entity_accuracy

# Example: Calculate CER between OCR output and ground truth
print("=" * 60)
print("EVALUATION METRICS DEMO")
print("=" * 60)

# Sample CER calculation
ground_truth_text = "AMOXICILLIN CAPSULES USP 500 mg"
ocr_text = "AMOXICILIN CAPSULES USP 500 mg"  # Missing one 'L'

cer = calculate_cer(ground_truth_text, ocr_text)
print(f"\nCER Example:")
print(f"  Ground Truth: {ground_truth_text!r}")
print(f"  OCR Output:   {ocr_text!r}")
print(f"  CER:          {cer:.4f} ({cer*100:.2f}%)")

# Sample Entity Match Rate calculation
gt_entities = {
    "drug_name": "Amoxicillin",
    "strength": "500 mg",
    "dosage_form": "Capsules"
}
pred_entities = {
    "drug_name": "Amoxicilin",  # Typo
    "strength": "500 mg",
    "dosage_form": "Capsules"
}

emr = calculate_entity_accuracy(gt_entities, pred_entities)
print(f"\nEntity Match Rate Example:")
print(f"  Ground Truth: {gt_entities}")
print(f"  Predicted:    {pred_entities}")
print(f"  EMR:          {emr:.2%}")

## Summary

This notebook demonstrated the full Intelligent Pharma-Context Engine pipeline:

1. **Detection & Extraction** (Stage 1)
   - Image preprocessing (CLAHE, bilateral filter, shadow removal)
   - Tesseract OCR with bounding boxes
   - Barcode/DataMatrix detection

2. **Verification** (Stage 2)
   - LLM-based entity extraction (Gemini)
   - Cross-reference with OpenFDA and RxNorm
   - Fuzzy matching for OCR errors
   - Confidence scoring

3. **Enrichment** (Stage 3)
   - Clinical context generation
   - Storage requirements, warnings, side effects
   - Human-readable summary

The output is a structured JSON record with full provenance tracking.